In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.stats import t as student_t

# ============================================================
# Paths
# ============================================================

ROOT = Path("..")
DML_RUN = Path("../outputs/dml")
RESULTS = DML_RUN / "dml_all_results.csv"
SCHEME = "local_p90"

OUTDIR = ROOT / "outputs" / "figures" / "fig2"
OUTDIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# Style and labels
# ============================================================

MARGINAL_MARK = "†"
SYMBOL_ONLY_FS = 8.0
SYMBOL_GAP = 0.035

COLORS = {
    "Building density": "#6A4A9C",
    "Mean building height": "#E34E6F",
}

TREATMENTS = ["Building density", "Mean building height"]

TREATMENT_LABELS = {
    "Building density": "Density",
    "Mean building height": "Height",
}

OUTCOMES = ["log_exp_day", "log_exp_night", "log_exp_compound"]

OUTCOME_LABELS = {
    "log_exp_day": "Daytime",
    "log_exp_night": "Nighttime",
    "log_exp_compound": "Compound",
}

CLIMATE_GROUPS = ["Arid (B)", "Tropical (A)", "Temperate/Highland (C)"]

CLIMATE_LABELS = {
    "Arid (B)": "Arid",
    "Tropical (A)": "Tropical",
    "Temperate/Highland (C)": "Temperate",
}

POP_OUTCOMES = ["log_pop"]

HWD_OUTCOMES = ["log_hwd_day", "log_hwd_night", "log_hwd_compound"]

COMPONENT_LABELS = {
    "log_pop": "Population",
    "log_hwd_day": "Day HWD",
    "log_hwd_night": "Night HWD",
    "log_hwd_compound": "Compound HWD",
}

plt.rcParams.update(plt.rcParamsDefault)
plt.rcParams.update(
    {
        "font.family": "Arial",
        "font.size": 8.5,
        "axes.linewidth": 0.8,
        "xtick.major.width": 0.8,
        "ytick.major.width": 0.8,
        "pdf.fonttype": 42,
        "svg.fonttype": "none",
    }
)

# ============================================================
# Helper functions
# ============================================================

def get_star(p):
    if pd.isna(p):
        return ""
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    if p < 0.10:
        return MARGINAL_MARK
    return ""


def value_label(theta, p, decimals=2, show_number=True):
    return (f"{theta:.{decimals}f}" if show_number else "") + get_star(p)


def clean_axis(ax):
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(axis="both", labelsize=8.5)


def add_zero_line(ax, axis="y"):
    if axis == "y":
        ax.axhline(0, color="#777777", lw=0.8, ls="--", zorder=1)
        ax.grid(axis="y", color="#E9E9E9", lw=0.75, zorder=0)
    else:
        ax.axvline(0, color="#777777", lw=0.8, ls="--", zorder=1)
        ax.grid(axis="x", color="#E9E9E9", lw=0.75, zorder=0)


def signed_yerr(theta, lo, hi):
    return np.array([[theta - lo], [hi - theta]])


def standardize_for_plot(df):
    df = df.copy()

    required = ["theta", "CI_lo_cluster", "CI_hi_cluster", "pval_cluster"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}")

    df["CI_lo"] = df["CI_lo_cluster"]
    df["CI_hi"] = df["CI_hi_cluster"]
    df["pval"] = df["pval_cluster"]

    se = df["SE_cluster"].astype(float)
    dof = df["n_clusters"] - 1
    t95 = student_t.ppf(0.975, dof)
    if not np.allclose(df["theta"] - t95 * se, df["CI_lo_cluster"], atol=1e-6):
        raise ValueError("The 95% CI recomputed from the standard error does not match the results file; check the degrees of freedom")
    t90 = student_t.ppf(0.95, dof)
    df["CI_lo_90"] = df["theta"] - t90 * se
    df["CI_hi_90"] = df["theta"] + t90 * se
    excl = (df["CI_lo_90"] > 0) | (df["CI_hi_90"] < 0)
    if not (excl == (df["pval"] < 0.10)).all():
        raise ValueError("The 90% CI is inconsistent with p < 0.10")

    return df


def add_koppen_from_blocks(df):
    df = df.copy()
    mapping = {}
    for zone in sorted(df["sample"].unique()):
        f = DML_RUN / "blocks" / f"{SCHEME}__climate_log1p__base__{zone}" / "climate_zone.txt"
        if not f.is_file():
            raise FileNotFoundError(f"Climate-group name file not found: {f}")
        mapping[zone] = f.read_text(encoding="utf-8").strip()
    df["koppen_broad"] = df["sample"].map(mapping)
    missing = sorted(set(CLIMATE_GROUPS) - set(df["koppen_broad"]))
    if missing:
        raise ValueError(f"Climate groups missing: {missing}; found {sorted(set(mapping.values()))}")
    return df


def get_row(data, treatment, outcome):
    sub = data[
        data["treatment_label"].eq(treatment)
        & data["outcome"].eq(outcome)
    ]

    if sub.empty:
        raise ValueError(f"Missing row: treatment={treatment}, outcome={outcome}")

    if len(sub) > 1:
        raise ValueError(f"Multiple rows found for {treatment}, {outcome}")

    return sub.iloc[0]


def plot_grouped_points(
    ax,
    data,
    title,
    ylim,
    show_ylabel=False,
    label_fs=6.5,
    show_numbers=True,
):
    x = np.arange(len(OUTCOMES))
    offsets = {
        "Building density": -0.11,
        "Mean building height": 0.11,
    }

    for treatment in TREATMENTS:
        color = COLORS[treatment]
        vals, lo_vals, hi_vals, pvals, lo2_vals, hi2_vals = [], [], [], [], [], []

        for outcome in OUTCOMES:
            row = get_row(data, treatment, outcome)
            vals.append(row["theta"])
            lo_vals.append(row["CI_lo"])
            hi_vals.append(row["CI_hi"])
            pvals.append(row["pval"])
            lo2_vals.append(row["CI_lo_90"])
            hi2_vals.append(row["CI_hi_90"])

        vals = np.asarray(vals, dtype=float)
        lo_vals = np.asarray(lo_vals, dtype=float)
        hi_vals = np.asarray(hi_vals, dtype=float)
        lo2_vals = np.asarray(lo2_vals, dtype=float)
        hi2_vals = np.asarray(hi2_vals, dtype=float)

        xpos = x + offsets[treatment]

        ax.errorbar(
            xpos,
            vals,
            yerr=np.vstack([vals - lo_vals, hi_vals - vals]),
            fmt="none",
            ecolor=color,
            elinewidth=1.05,
            capsize=2.5,
            capthick=1.05,
            zorder=3,
        )

        ax.errorbar(
            xpos,
            vals,
            yerr=np.vstack([vals - lo2_vals, hi2_vals - vals]),
            fmt="none",
            ecolor=color,
            elinewidth=2.8,
            capsize=0,
            zorder=3.1,
        )

        ax.plot(xpos, vals, "o", ms=5.0, color=color, mec=color, mew=1.0, zorder=4)

        span = ylim[1] - ylim[0]
        top_vals = np.maximum(hi_vals, hi2_vals)
        for xi, v, hi, pv in zip(xpos, vals, top_vals, pvals):
            ax.text(
                xi,
                hi + (0.025 if show_numbers else SYMBOL_GAP) * span,
                value_label(v, pv, show_number=show_numbers),
                ha="center",
                va="bottom",
                fontsize=label_fs if show_numbers else SYMBOL_ONLY_FS,
                color="#444444",
                linespacing=1.05,
                zorder=5,
            )

    add_zero_line(ax, "y")
    clean_axis(ax)

    ax.set_title(title, loc="left", fontsize=10.4, fontweight="bold", pad=6)
    ax.set_xticks(x)
    ax.set_xticklabels([OUTCOME_LABELS[o] for o in OUTCOMES])
    ax.set_xlim(-0.474, 2.474)
    ax.set_ylim(*ylim)

    if show_ylabel:
        ax.set_ylabel(r"DML estimate ($\theta$)")


def plot_component_forest(ax, data, outcomes, title, xlim, decimals=3, show_numbers=True):
    ypos = np.arange(len(outcomes))[::-1]
    ymap = {outcome: y for outcome, y in zip(outcomes, ypos)}

    offsets = {
        "Building density": 0.10,
        "Mean building height": -0.10,
    }
    pad = (0.012 if show_numbers else SYMBOL_GAP * 0.6) * (xlim[1] - xlim[0])

    for treatment in TREATMENTS:
        sub = data[data["treatment_label"].eq(treatment)]
        color = COLORS[treatment]

        for _, row in sub.iterrows():
            if row["outcome"] not in ymap:
                continue

            xv = row["theta"]
            yv = ymap[row["outcome"]] + offsets[treatment]
            pv = row["pval"]

            ax.errorbar(
                xv,
                yv,
                xerr=signed_yerr(xv, row["CI_lo"], row["CI_hi"]),
                fmt="o",
                ms=5.0,
                color=color,
                mfc=color,
                mec=color,
                mew=1.0,
                ecolor=color,
                elinewidth=1.05,
                capsize=2.6,
                zorder=3,
            )

            ax.plot(
                [row["CI_lo_90"], row["CI_hi_90"]],
                [yv, yv],
                color=color,
                lw=2.8,
                solid_capstyle="butt",
                zorder=2.9,
            )
            lo_out = min(row["CI_lo"], row["CI_lo_90"])
            hi_out = max(row["CI_hi"], row["CI_hi_90"])

            on_left = hi_out > xlim[0] + 0.62 * (xlim[1] - xlim[0])
            ax.text(
                lo_out - pad if on_left else hi_out + pad,
                yv - (0.0 if show_numbers else 0.035),
                value_label(xv, pv, decimals=decimals, show_number=show_numbers),
                ha="right" if on_left else "left",
                va="center",
                fontsize=6.3 if show_numbers else SYMBOL_ONLY_FS,
                color=color,
                zorder=4,
            )

    add_zero_line(ax, "x")
    clean_axis(ax)

    ax.set_title(title, loc="left", fontsize=10.4, fontweight="bold", pad=6)
    ax.set_yticks(ypos)
    ax.set_yticklabels([COMPONENT_LABELS[o] for o in outcomes])
    ax.set_xlabel(r"DML estimate ($\theta$)")
    ax.set_xlim(*xlim)
    ax.set_ylim(-0.6, len(outcomes) - 0.4)
    ax.tick_params(axis="y", length=0)


# ============================================================
# Load and prepare data
# ============================================================

all_raw = pd.read_csv(RESULTS)
all_raw = all_raw[all_raw["scheme"].eq(SCHEME) & all_raw["w_variant"].eq("base")].copy()
if "status" in all_raw.columns:
    bad = all_raw[~all_raw["status"].astype(str).str.lower().isin(["ok", "completed", "success"])]
    if len(bad) and bad["theta"].isna().any():
        raise ValueError("The results contain unfinished estimates; check the DML output")

all_raw["outcome_group"] = np.select(
    [all_raw["outcome"].str.startswith("log_exp"), all_raw["outcome"].eq("log_pop"),
     all_raw["outcome"].str.startswith("log_hwd")],
    ["exposure", "population", "hwd"], default="other")

main_raw = all_raw[all_raw["analysis"].eq("main_log1p")].copy()
hte_raw = all_raw[all_raw["analysis"].eq("climate_log1p")].copy()

main_plot = standardize_for_plot(main_raw)
hte_plot = standardize_for_plot(hte_raw)
hte_plot = add_koppen_from_blocks(hte_plot)

main_base = main_plot[main_plot["w_variant"].eq("base")].copy()

main_exp = main_base[
    main_base["outcome_group"].eq("exposure")
    & main_base["outcome"].isin(OUTCOMES)
    & main_base["treatment_label"].isin(TREATMENTS)
].copy()

component_df = main_base[
    main_base["outcome"].isin(POP_OUTCOMES + HWD_OUTCOMES)
    & main_base["treatment_label"].isin(TREATMENTS)
].copy()

hte_exp = hte_plot[
    hte_plot["outcome_group"].eq("exposure")
    & hte_plot["outcome"].isin(OUTCOMES)
    & hte_plot["treatment_label"].isin(TREATMENTS)
    & hte_plot["koppen_broad"].isin(CLIMATE_GROUPS)
].copy()

# ============================================================
# Draw figure
# ============================================================

fig = plt.figure(figsize=(8.4, 6.8), dpi=240)

gs = fig.add_gridspec(
    3,
    3,
    height_ratios=[0.95, 1.10, 1.15],
    hspace=0.68,
    wspace=0.34,
)

ax_a = fig.add_subplot(gs[0, :])

plot_grouped_points(
    ax_a,
    main_exp,
    "(a) Full-sample exposure estimates",
    ylim=(-0.60, 1.10),
    show_ylabel=True,
)

legend_handles = [
    Line2D(
        [0],
        [0],
        marker="s",
        linestyle="none",
        markerfacecolor=COLORS[t],
        markeredgecolor="white",
        markeredgewidth=0.6,
        markersize=8,
        label=TREATMENT_LABELS[t],
    )
    for t in TREATMENTS
]

ax_a.legend(
    handles=legend_handles,
    frameon=False,
    loc="upper right",
    bbox_to_anchor=(1.0, 1.0),
    borderaxespad=0,
    handletextpad=0.45,
    ncol=2,
)

gs_b = gs[1, :].subgridspec(
    1,
    2,
    width_ratios=[0.35, 0.65],
    wspace=0.28,
)

ax_b1 = fig.add_subplot(gs_b[0, 0])
ax_b2 = fig.add_subplot(gs_b[0, 1])

plot_component_forest(
    ax_b1,
    component_df,
    POP_OUTCOMES,
    "(b) Population component",
    xlim=(-0.15, 1.02),
    show_numbers=False,
)

plot_component_forest(
    ax_b2,
    component_df,
    HWD_OUTCOMES,
    "Heatwave-day component",
    xlim=(-0.18, 0.18),
    show_numbers=False,
)

axes_c = [fig.add_subplot(gs[2, i]) for i in range(3)]

for ax, group in zip(axes_c, CLIMATE_GROUPS):
    sub = hte_exp[hte_exp["koppen_broad"].eq(group)].copy()

    title = (
        f"(c) {CLIMATE_LABELS[group]}"
        if ax is axes_c[0]
        else CLIMATE_LABELS[group]
    )

    plot_grouped_points(
        ax,
        sub,
        title,
        ylim=(-1.15, 2.35),
        show_ylabel=(ax is axes_c[0]),
        label_fs=6.0,
        show_numbers=False,
    )

fig.subplots_adjust(
    left=0.08,
    right=0.985,
    bottom=0.12,
    top=0.94,
)

fig.text(
    0.08,
    0.035,
    "Error bars: city-clustered 95% (thin, capped) and 90% (thick) confidence intervals."
    f"  *** p < 0.001,  ** p < 0.01,  * p < 0.05,  {MARGINAL_MARK} p < 0.10 (city-clustered).",
    ha="left",
    va="bottom",
    fontsize=6.8,
    color="#444444",
)

# ============================================================
# Save
# ============================================================

png_path = OUTDIR / "fig2.png"
pdf_path = OUTDIR / "fig2.pdf"
svg_path = OUTDIR / "fig2.svg"
table_path = OUTDIR / "fig2_plotted_values.csv"

fig.savefig(png_path, dpi=300, bbox_inches="tight")
fig.savefig(pdf_path, bbox_inches="tight")
fig.savefig(svg_path, bbox_inches="tight")

cols = ["treatment_label", "outcome", "n", "n_cities", "theta", "CI_lo_cluster", "CI_hi_cluster", "pval_cluster",
        "CI_lo_iid", "CI_hi_iid", "pval_iid", "pval"]
plotted = pd.concat([
    main_exp.assign(panel="a", group="Full sample"),
    component_df.assign(panel="b", group="Full sample"),
    hte_exp.assign(panel="c", group=hte_exp["koppen_broad"]),
])[["panel", "group"] + cols]
plotted["marker"] = plotted["pval"].map(get_star)
plotted = plotted.drop(columns="pval")
src = pd.concat([main_exp, component_df, hte_exp])
plotted["CI90_lo_cluster"] = src["CI_lo_90"].to_numpy()
plotted["CI90_hi_cluster"] = src["CI_hi_90"].to_numpy()
plotted.to_csv(table_path, index=False, encoding="utf-8-sig")


plt.show()

print("Output:", OUTDIR)